Etapa 2 — Entender o comportamentoComo esses 

_**Como esses clientes utilizam o banco?**_


**Analisar:**

- frequência de transações
- valor das transações
- tipos de transação
- recência
- comportamento por região
- comportamento por faixa etária
- comportamento por gênero
- evolução temporal, se houver data suficiente


In [0]:
df = spark.table("workspace.default.bank_transactions").toPandas()

In [0]:
df.rename(columns={
    "TransactionID": "ID_Transacao",
    "CustomerID": "ID_Cliente",
    "CustomerDOB": "Data_Nascimento",
    "CustGender": "Genero",
    "CustLocation": "Localizacao",
    "CustAccountBalance": "Saldo_Conta",
    "TransactionDate": "Data_Transacao",
    "TransactionTime": "Horario_Transacao",
    "TransactionAmount (INR)": "Valor_Transacao"
}, inplace=True)

In [0]:
cliente = (
    df.groupby("ID_Cliente")
      .agg(
          qtd_transacoes=("ID_Transacao", "count"),
          volume_total=("Valor_Transacao", "sum"),
          ticket_medio=("Valor_Transacao", "mean"),
          saldo_medio=("Saldo_Conta", "mean"),
          primeira_transacao=("Data_Transacao", "min"),
          ultima_transacao=("Data_Transacao", "max")
      )
      .reset_index()
)

cliente.head()

In [0]:
cliente.shape

In [0]:
cliente.describe()

Cliente típico
1 transação
aproximadamente R$ 537 movimentados
ticket típico de aproximadamente R$ 500
saldo de aproximadamente R$ 18,7 mil
Mas a média da carteira mostra:
1,19 transações
R$ 1.867 movimentados
ticket de R$ 1.575
saldo de R$ 115 mil

Essa diferença é o principal achado.

A carteira apresenta baixa frequência de utilização e forte assimetria financeira. O cliente típico realizou apenas uma transação e movimentou cerca de R$ 537, enquanto poucos clientes com operações e saldos muito elevados elevam significativamente as médias da carteira.

In [0]:
cliente[
    ["qtd_transacoes", "volume_total", "ticket_medio", "saldo_medio"]
].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99])

Pelo menos 75% da distribuição está concentrada em clientes com apenas uma transação.
Isso é interessante porque o aumento do volume não parece depender de uma quantidade muito grande de operações.

In [0]:
cliente[
    ["qtd_transacoes", "volume_total", "ticket_medio", "saldo_medio"]
].corr()

O aumento do volume não parece estar sendo explicado pela frequência de transações.

Em outras palavras, um cliente fazer mais transações não significa necessariamente que ele terá um volume muito maior.

In [0]:
cliente.sort_values(
    "volume_total",
    ascending=False
).head(20)

Os valores extremos de volume estão concentrados principalmente em clientes de baixa frequência transacional. Entre os maiores outliers analisados, a maioria realizou apenas uma operação, mas com valores superiores a R$ 500 mil. A baixa correlação entre quantidade de transações e volume total (r = 0,098), combinada à forte correlação entre volume total e ticket médio (r = 0,960), indica que os valores extremos são mais associados ao tamanho das operações do que à recorrência dos clientes.

In [0]:
cliente.sort_values(
    "saldo_medio",
    ascending=False
).head(20)

In [0]:
cliente.groupby("qtd_transacoes").agg(
    clientes=("ID_Cliente", "count"),
    volume_medio=("volume_total", "mean"),
    volume_mediano=("volume_total", "median"),
    ticket_medio=("ticket_medio", "mean"),
    saldo_medio=("saldo_medio", "mean")
).reset_index()

A utilização do banco é predominantemente pontual: 83,8% dos clientes realizaram apenas uma transação e 97,9% realizaram no máximo duas. O volume movimentado cresce praticamente de forma proporcional à frequência, enquanto o ticket médio permanece estável em torno de R$ 1,57 mil. Isso indica que, na base analisada, clientes mais frequentes movimentam mais principalmente porque transacionam mais vezes, e não porque realizam operações significativamente maiores.

In [0]:
p99 = cliente["volume_total"].quantile(0.99)

p99

In [0]:
outliers = cliente[
    cliente["volume_total"] >= p99
].copy()

outliers["qtd_transacoes"].value_counts().sort_index()

In [0]:
outliers.groupby("qtd_transacoes").agg(
    clientes=("ID_Cliente", "count"),
    volume_medio=("volume_total", "mean"),
    volume_total=("volume_total", "sum")
).reset_index()

In [0]:
outliers["saldo_medio"].value_counts().head(20)

In [0]:
outliers.groupby("saldo_medio").agg(
    clientes=("ID_Cliente", "nunique"),
    volume_medio=("volume_total", "mean"),
    volume_total=("volume_total", "sum"),
    ticket_medio=("ticket_medio", "mean")
).sort_values("clientes", ascending=False).head(20)

In [0]:
outliers["volume_sobre_saldo"] = (
    outliers["volume_total"] / outliers["saldo_medio"]
)

In [0]:
outliers["volume_sobre_saldo"].describe(
    percentiles=[.25, .5, .75, .9, .95, .99]
)

In [0]:
outliers.sort_values(
    "volume_sobre_saldo",
    ascending=False
)[[
    "ID_Cliente",
    "qtd_transacoes",
    "volume_total",
    "saldo_medio",
    "volume_sobre_saldo"
]].head(20)